## E7 G1 — Original label missingness and clock feasibility

G0 verified all eight original CIC-IDS2017 CSVs, 3,119,345 rows. Thursday WebAttacks has 288,602 missing labels and ambiguous source encoding. This phase performs read-only full CSV source validation; never relabels, trains, assumes related flow chronology, touches frozen UNSW tests or exports IP/flow rows. Completed G1 aggregate reports are immediately saved to GitHub create-only and independently verified.

In [ ]:
from pathlib import Path
import hashlib,json,importlib.util
from urllib.request import urlopen

REPO="itsCodeBakery/rare-attack-spatiotemporal-ids"
SRC_COMMIT="b7a175fd41216f53a701700e65e88d248d15b588"
SRC_PATH="research/E7_CICIDS2017/scripts/e7_g1_label_clock_diagnostic.py"
SRC_SHA="7b9acf476f60d67d2b8a8763d28007f4257e419d"
REG_COMMIT="99a1782951ace3589a4b13a998e05411b30b80cc"
REG_SHA="3e1c5f2e3252fa6923108eba02ac238b5a8561dc"
REG_PATH="research/E7_CICIDS2017/configs/e7_g0_timestamp_registry_001.json"
WORK=Path("/kaggle/working/E7_G1_LABEL_CLOCK_001")
WORK.mkdir(parents=True,exist_ok=True)

def blob_sha(b):
    return hashlib.sha1(b"blob "+str(len(b)).encode()+bytes([0])+b).hexdigest()

def pinned(commit,path,sha):
    url="https://raw.githubusercontent.com/"+REPO+"/"+commit+"/"+path
    with urlopen(url,timeout=90) as response: data=response.read()
    assert blob_sha(data)==sha,"Remote source revision mismatch; halt"
    return data

code=pinned(SRC_COMMIT,SRC_PATH,SRC_SHA)
module_path=WORK/"e7_g1_label_clock_diagnostic.py"
if module_path.exists():
    assert module_path.read_bytes()==code,"Cached G1 script differs; halt"
else:module_path.write_bytes(code)
spec=importlib.util.spec_from_file_location("g1_audit",module_path)
g1=importlib.util.module_from_spec(spec)
spec.loader.exec_module(g1)
registry=json.loads(pinned(REG_COMMIT,REG_PATH,REG_SHA))
DATA=Path("/kaggle/input/datasets/dsshayan/dscicids")
files=sorted(DATA.rglob("*.csv"))
assert len(files)==8 and {p.name for p in files}==set(registry["filename_registry"])
assert len({str(p.parent) for p in files})==1
assert files[0].parent.name=="TrafficLabelling "
print("G1 SOURCE PREFLIGHT PASSED: 8 external CSVs, pinned scripts; no Git commands.")

In [ ]:
import base64,time,re
from urllib.error import HTTPError
from urllib.parse import quote
from urllib.request import Request
from kaggle_secrets import UserSecretsClient

token=UserSecretsClient().get_secret("pushRare")
assert token,"Set Kaggle pushRare secret, Internet ON."
def api(method,url,payload=None):
    head={"Authorization":"Bearer "+token,"Accept":"application/vnd.github+json",
          "X-GitHub-Api-Version":"2022-11-28"}
    data=None if payload is None else json.dumps(payload).encode()
    if data is not None:head["Content-Type"]="application/json"
    with urlopen(Request(url,data=data,headers=head,method=method),timeout=70) as r:
        return json.load(r)
BASE="https://api.github.com/repos/"+REPO+"/contents/"
PREFIX="research/E7_CICIDS2017/results/G1_LABEL_CLOCK"
def remote_bytes(url):
    return base64.b64decode(api("GET",url)["content"])
def remote_check(url,data,blob=None,commit=None):
    refs=[]
    if blob:refs.append("https://api.github.com/repos/"+REPO+"/git/blobs/"+blob)
    if commit:refs.append(url+"?ref="+quote(commit))
    refs.append(url)
    for ref in refs:
        try:b=remote_bytes(ref)
        except HTTPError as e:
            if e.code in (404,409,429,500,502,503,504):continue
            raise
        if b!=data:raise RuntimeError("Existing remote content differs: no overwrite")
        return True
    return False
def send_report(name,r):
    assert name==r["source_file"] and name in registry["filename_registry"]
    assert r["schema"]=="E7_G1_LABEL_CLOCK_DIAGNOSTIC_001"
    assert r["causal_order_proven"] is False and r["training_authorized"] is False
    assert r["trusted_original_labels"] is False
    assert re.fullmatch("[a-f0-9]{64}",r["original_source_sha256"])
    raw=(json.dumps(r,sort_keys=True,indent=2,ensure_ascii=False)+"\n").encode()
    local=WORK/(name+".g1.json")
    if local.exists():assert local.read_bytes()==raw,"Local checkpoint mismatch"
    else:local.write_bytes(raw)
    url=BASE+quote(PREFIX+"/"+name+".g1.json",safe="/")
    if remote_check(url,raw):return "ALREADY_VERIFIED"
    try:
        put=api("PUT",url,{"message":"research(e7-g1): original-source aggregate "+name,
                   "branch":"main","content":base64.b64encode(raw).decode()})
    except HTTPError as e:
        if e.code not in (409,422):raise
        for sec in (0,1,2,3,5,8):
            if sec:time.sleep(sec)
            if remote_check(url,raw):return "VERIFIED_CONCURRENT"
        raise RuntimeError("Remote conflict; original evidence unchanged") from e
    blob=put["content"]["sha"];commit=put["commit"]["sha"]
    for sec in (0,1,2,3,5,8,13):
        if sec:time.sleep(sec)
        if remote_check(url,raw,blob,commit):return "UPLOADED_VERIFIED"
    raise RuntimeError("PUT accepted but not visible yet; do not resend blindly")

G0={}
ENC={}
for p in files:
    url=BASE+"research/E7_CICIDS2017/results/G0/"+quote(p.name+".audit.json")
    item=json.loads(remote_bytes(url))
    assert item["input_artifact_sha256"]==g1.digest(p),"Original CSV bytes changed"
    assert item["timestamp_format_declared"]==registry["filename_registry"][p.name]["timestamp_format"]
    G0[p.name]=item
    ENC[p.name]="cp1252" if any("using cp1252" in gate for gate in item["explicit_followup_gates"]) else "utf-8-sig"
    print("PREVIOUS G0 VALIDATED:",p.name,ENC[p.name])
print("G1 PREFLIGHT PASSED. Running original source row-quality analyses.")

In [ ]:
all_reports=[]
for i,p in enumerate(files,1):
    print("\nG1 ORIGINAL SOURCE",i,"/8:",p.name,flush=True)
    cfg=registry["filename_registry"][p.name]
    r=g1.inspect(p,timestamp_format=cfg["timestamp_format"],
                 expected_day=cfg["expected_weekday"],encoding=ENC[p.name])
    original=G0[p.name]
    assert r["rows"]==original["total_rows_inspected"]
    assert r["original_source_sha256"]==original["input_artifact_sha256"]
    def canonical_labels(mapping):
        result={}
        for k,v in mapping.items():
            key="__MISSING_LABEL__" if k in ("__MISSING__","__MISSING_LABEL__") else k
            result[key]=result.get(key,0)+int(v)
        return result
    assert canonical_labels(r["labels"])==canonical_labels(original["files"][0]["labels"]),(
        "True source label drift: stop, preserve evidence"
    )
    state=send_report(p.name,r)
    print("GITHUB G1:",state,"| rows:",r["rows"],
          "| missing:",r["missing_label_rows"],
          "| missing runs:",r["missing_label_runs_in_csv_row_order"],
          "| time reversals:",r["source_adjacent_time_reversals"],flush=True)
    all_reports.append(r)
assert len(all_reports)==8
print("\nE7 G1 COMPLETE: 8/8 SOURCE DIAGNOSTICS SAVED ON GITHUB")
print("ALL ROWS:",sum(q["rows"] for q in all_reports))
print("MISSING LABELS:",sum(q["missing_label_rows"] for q in all_reports))
print("PUBLIC REPORTS: https://github.com/"+REPO+"/tree/main/"+PREFIX)
print("SOURCE-ONLY. No training. No causal-order or trustworthy-label claims.")